In [1]:
import numpy as np

# 1. Устройство механизма внимания

## 1.1. В numpy

In [2]:
# 1. Исходные эмбеддинги для входящих токенов
# 3 слова в предложении х 4 элемента эмбеддинга
X = np.array([[1, 0, 1, 0], [2, 2, 2, 2], [0, 1, 0, 1]])
print(X)

[[1 0 1 0]
 [2 2 2 2]
 [0 1 0 1]]


In [5]:
# 2. Матрицы весов. Настраиваются в процессе обучения
# Взяты произвольные числа для примера
np.random.seed(42)
W_Q = np.random.randn(4,3)
W_K = np.random.randn(4,3)
W_V = np.random.randn(4,3)
print("WQ:\n", W_Q)
print("\nWK:\n", W_K)
print("\nWV:\n", W_V)

WQ:
 [[ 0.49671415 -0.1382643   0.64768854]
 [ 1.52302986 -0.23415337 -0.23413696]
 [ 1.57921282  0.76743473 -0.46947439]
 [ 0.54256004 -0.46341769 -0.46572975]]

WK:
 [[ 0.24196227 -1.91328024 -1.72491783]
 [-0.56228753 -1.01283112  0.31424733]
 [-0.90802408 -1.4123037   1.46564877]
 [-0.2257763   0.0675282  -1.42474819]]

WV:
 [[-0.54438272  0.11092259 -1.15099358]
 [ 0.37569802 -0.60063869 -0.29169375]
 [-0.60170661  1.85227818 -0.01349722]
 [-1.05771093  0.82254491 -1.22084365]]


In [6]:
# 3. Рассчитываем проекции входящих токенов
# Query, key, value
Q = X@W_Q
K = X@W_K
V = X@W_V
print("Q:\n", Q)
print("\nK:\n", K)
print("\nV:\n", V)

Q:
 [[ 2.07592697  0.62917043  0.17821415]
 [ 8.28303374 -0.13680128 -1.04330512]
 [ 2.0655899  -0.69757107 -0.69986671]]

K:
 [[-0.6660618  -3.32558395 -0.25926906]
 [-2.90825127 -8.54177372 -2.73953983]
 [-0.78806383 -0.94530292 -1.11050085]]

V:
 [[-1.14608934  1.96320077 -1.1644908 ]
 [-3.65620449  4.37021399 -5.3540564 ]
 [-0.68201291  0.22190622 -1.5125374 ]]


In [7]:
# 4. Рассчитываем матрицу внимания:
scores = Q@K.T
print(scores)

[[ -3.52126015 -11.89977344  -2.42862657]
 [ -4.79157151 -20.06244187  -5.23964942]
 [  1.1254744    1.8685525   -0.19119814]]


In [8]:
# 5. Масштабируем:
norm_scores = Q@K.T/(K.shape[0]**0.5)
print(norm_scores)

[[ -2.0330005   -6.8703374   -1.4021682 ]
 [ -2.7664151  -11.58305621  -3.025113  ]
 [  0.64979294   1.07880929  -0.1103883 ]]


In [9]:
# 6. Рассчитываем веса внимания
def softmax_2d(scores):
    max_per_row = np.max(scores, axis=1, keepdims=True)
    shifted = scores - max_per_row
    exp_scores = np.exp(shifted)
    row_sums = np.sum(exp_scores, axis=1, keepdims=True)
    return exp_scores / row_sums

attention_weights = np.round(softmax_2d(norm_scores), 3)
print(attention_weights)

[[0.346 0.003 0.651]
 [0.564 0.    0.436]
 [0.333 0.511 0.156]]


In [10]:
# 7. Представление предложения после обработки механизмом внимания
output = attention_weights@V
print(output)

[[-0.85150593  0.83683906 -1.40363783]
 [-0.94375201  1.20399635 -1.31623912]
 [-2.35636226  2.92154258 -3.35965409]]


# 1.2. В PyTorch

In [11]:
import torch
import torch.nn.functional as F
import numpy as np

In [12]:
# 1. Исходные эмбеддинги для входящих токенов
# 3 слова в предложении х 4 элемента эмбеддинга

X = np.array([
    [1, 0, 1, 0],
    [2, 2, 2, 2],
    [0, 1, 0, 1]
])

X = torch.tensor(X, dtype=torch.float32)

print("X:")
print(X)

X:
tensor([[1., 0., 1., 0.],
        [2., 2., 2., 2.],
        [0., 1., 0., 1.]])


In [13]:
# 2. Матрицы весов. Настраиваются в процессе обучения
# Взяты произвольные числа для примера
W_Q = torch.randn(4, 3)
W_K = torch.randn(4, 3)
W_V = torch.randn(4, 3)
print("WQ:\n", W_Q)
print("\nWK:\n", W_K)
print("\nWV:\n", W_V)

WQ:
 tensor([[-0.6669, -0.7601,  1.0985],
        [ 0.6099, -0.8367, -0.1614],
        [ 0.5242,  0.8963, -0.1993],
        [-0.6067, -0.0901, -0.0369]])

WK:
 tensor([[-0.1650, -1.7283,  0.0023],
        [ 0.3835, -0.7282,  0.8879],
        [-0.2263, -0.4398,  0.8349],
        [ 0.1822,  0.6179, -0.2423]])

WV:
 tensor([[ 1.0138, -1.0495,  2.0411],
        [-1.1850, -0.9532,  0.0670],
        [ 0.1624, -0.8910, -0.3743],
        [ 1.5927,  1.2115, -0.2391]])


In [14]:
# 3. Рассчитываем проекции входящих токенов
# Query, key, value
Q = X@W_Q
K = X@W_K
V = X@W_V
print("Q:\n", Q)
print("\nK:\n", K)
print("\nV:\n", V)

Q:
 tensor([[-0.1428,  0.1362,  0.8992],
        [-0.2791, -1.5811,  1.4019],
        [ 0.0032, -0.9268, -0.1982]])

K:
 tensor([[-0.3913, -2.1681,  0.8372],
        [ 0.3490, -4.5568,  2.9656],
        [ 0.5657, -0.1103,  0.6456]])

V:
 tensor([[ 1.1762, -1.9405,  1.6668],
        [ 3.1679, -3.3645,  2.9894],
        [ 0.4078,  0.2582, -0.1721]])


In [15]:
# 4. Рассчитываем матрицу внимания:
scores = Q@K.T
print(scores)

tensor([[ 0.5134,  1.9963,  0.4847],
        [ 4.7110, 11.2651,  0.9215],
        [ 1.8421,  3.6362, -0.0240]])


In [16]:
# 5. Масштабируем:
d_k = K.shape[-1]
norm_scores = scores / np.sqrt(d_k)
print(norm_scores)

tensor([[ 0.2964,  1.1526,  0.2799],
        [ 2.7199,  6.5039,  0.5321],
        [ 1.0635,  2.0994, -0.0138]])


In [17]:
# 6. Рассчитываем веса внимания
attention_weights = F.softmax(norm_scores,dim=-1)
print(attention_weights)

tensor([[0.2305, 0.5427, 0.2268],
        [0.0222, 0.9753, 0.0025],
        [0.2405, 0.6776, 0.0819]])


In [18]:
# 7. Представление предложения после обработки механизмом внимания
output = attention_weights @ V
print(output)

tensor([[ 2.0829, -2.2147,  1.9676],
        [ 3.1169, -3.3239,  2.9522],
        [ 2.4629, -2.7254,  2.4124]])


# 2. Простая модель внимания для классификации

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [20]:
# Загружаем игрушечный датасет
data = [
    ("good movie", 1),
    ("great film", 1),
    ("awesome acting", 1),
    ("bad movie", 0),
    ("terrible film", 0),
    ("boring acting", 0),
]

In [21]:
# Проводим простую векторизацию
vocab = {"good":0, "movie":1, "great":2, "film":3, "awesome":4, "acting":5, "bad":6, "terrible":7, "boring":8}

In [22]:
# Кодируем предложения в векторы
def encode(sentence):
    return torch.tensor([vocab[word] for word in sentence.split()], dtype=torch.long)

# Пример
print("Оригинал:", data[0][0])
print("Вектор:", encode(data[0][0]))

Оригинал: good movie
Вектор: tensor([0, 1])


In [23]:
X = [encode(s) for s, _ in data]
y = torch.tensor([label for _, label in data])

In [27]:
class SimpleAttention(nn.Module):
    def forward(self, X):
        # Для простоты обойдемся без дополнительных матриц W_Q, W_K, W_V
        Q = X
        K = X
        V = X
        
        # QK^T
        scores = Q @ K.T

        # масштабирование
        d_k = Q.shape[-1]

        scores = scores / (d_k ** 0.5)

        # Softmax
        attention_weights = F.softmax(scores,dim=-1)

        # Взвешенная сумма V
        output = attention_weights @ V

        return output, attention_weights

In [28]:
class AttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        # 1. Эмбеддинг
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 2. Внимание
        self.attention = SimpleAttention()

        # 3. Классификатор
        self.classifier = nn.Linear(embedding_dim, 1)

    def forward(self, tokens):
        # tokens:
        # [token_1, token_2, ...]

        # Эмбеддинг
        X = self.embedding(tokens)

        # Внимание
        attention_output, attention_weights = self.attention(X)

        # После Attention остается один вектор на каждый токен
        # Но для классификации предложения нам нужен один вектор на предложение
        # Применим простейший способ — усредним представления токенов:
        sentence_vector = attention_output.mean(dim=0)

        # Классификация
        logit = self.classifier(sentence_vector)

        return logit.squeeze(), attention_weights

In [29]:
model = AttentionClassifier(vocab_size=len(vocab),embedding_dim=4)
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)
loss_function = nn.BCEWithLogitsLoss()

In [30]:
for epoch in range(500):
    total_loss = 0
    for text, label in data:
        # Токенизация
        tokens = torch.tensor(encode(text),dtype=torch.long)

        # Прямой проход
        logit, attention_weights = model(tokens)

        # Целевая переменная
        target = torch.tensor(label, dtype=torch.float32)

        # Loss
        loss = loss_function(logit,target)

        # Обратный проход
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch}, "
            f"Loss = {total_loss:.4f}"
        )

C:\Users\D\AppData\Local\Temp\ipykernel_2732\586174822.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tokens = torch.tensor(encode(text),dtype=torch.long)


Epoch 0, Loss = 4.5658
Epoch 100, Loss = 0.0471
Epoch 200, Loss = 0.0085
Epoch 300, Loss = 0.0034
Epoch 400, Loss = 0.0017


In [31]:
test_sentences = ["great movie", "terrible acting", "awesome film", "boring movie"]
model.eval()
with torch.no_grad():
    for text in test_sentences:
        tokens = torch.tensor(encode(text), dtype=torch.long)
        logit, attention_weights = model(tokens)
        probability = torch.sigmoid(logit)
        prediction = int(probability >= 0.5)

        print(
            f"{text:20s}"
            f" probability={probability.item():.3f}"
            f" prediction={prediction}"
        )

great movie          probability=1.000 prediction=1
terrible acting      probability=0.000 prediction=0
awesome film         probability=1.000 prediction=1
boring movie         probability=0.000 prediction=0


C:\Users\D\AppData\Local\Temp\ipykernel_2732\3043968135.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tokens = torch.tensor(encode(text), dtype=torch.long)


# 3. Обучаем модель машинного перевода на трансформере

In [ ]:
#!pip install torch torchtext spacy datasets

In [50]:
import math
import time
import spacy

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from datasets import load_dataset
import spacy
from collections import Counter

In [51]:
# Загружаем датасет
train_data = load_dataset("bentrevett/multi30k", split = "train")
valid_data = load_dataset("bentrevett/multi30k", split = "validation")
test_data = load_dataset("bentrevett/multi30k", split = "test")

README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

C:\Users\D\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\D\.cache\huggingface\hub\datasets--bentrevett--multi30k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.jsonl:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

val.jsonl:   0%|          | 0.00/164k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/156k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [53]:
# Смотрим пример
print(train_data[0])

{'en': 'Two young, White males are outside near many bushes.', 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}


In [ ]:
# Для токенизации установим модели spaCy:
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm

In [55]:
# Сделаем максимально простую токенизацию с помощью spaCy
spacy_de = spacy.load("de_core_news_sm")
spacy_en = spacy.load("en_core_web_sm")

def tokenize_de(text):
    return [tok.text.lower() for tok in spacy_de(text)]

def tokenize_en(text):
    return [tok.text.lower() for tok in spacy_en(text)]

In [57]:
# Например
text = "Ein Mann sitzt auf einer Bank."
print(tokenize_de(text))
text = "A man is sitting on a bench."
print(tokenize_en(text))

['ein', 'mann', 'sitzt', 'auf', 'einer', 'bank', '.']
['a', 'man', 'is', 'sitting', 'on', 'a', 'bench', '.']


In [139]:
# Специальные токены
PAD, UNK, BOS, EOS = "<pad>", "<unk>", "<bos>", "<eos>"
PAD_IDX, UNK_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3

In [59]:
# Создаем словарь
class Vocabulary:
    def __init__(self):
        self.itos = [PAD,UNK,BOS,EOS]
        self.stoi = {token: i for i, token in enumerate(self.itos)}

    def build(self, tokenized_sentences, min_freq=2):
        counter = Counter()
        for tokens in tokenized_sentences:
            counter.update(tokens)

        for token, freq in counter.items():
            if freq >= min_freq:
                if token not in self.stoi:
                    self.stoi[token] = len(self.itos)
                    self.itos.append(token)
                    
    def numericalize(self, tokens):
        return [self.stoi.get(token,self.stoi[UNK]) for token in tokens]

    def __len__(self):
        return len(self.itos)

In [62]:
# Внимание, долгая процедура! Может занять несколько десятков минут и сожрать много памяти
de_tokens = [tokenize_de(example["de"]) for example in train_data]
en_tokens = [tokenize_en(example["en"]) for example in train_data]

In [140]:
# Создаем словарь
de_vocab = Vocabulary()
en_vocab = Vocabulary()

de_vocab.build(de_tokens, min_freq=2)
en_vocab.build(en_tokens, min_freq=2)

In [78]:
# Сохраняем построенные словари
working_dir = "C:\\Courses\\Innopolis\\2026\\ИИ Инженер\\Models\\"
torch.save(de_tokens, working_dir + "de_tokens.pt")
torch.save(en_tokens, working_dir + "en_tokens.pt")
torch.save({"itos": de_vocab.itos, "stoi": de_vocab.stoi}, working_dir + "de_vocab.pt")
torch.save({"itos": en_vocab.itos, "stoi": en_vocab.stoi}, working_dir + "en_vocab.pt")

In [ ]:
# Если затем нужно загрузить, раскомментируйте и запустите
# de_tokens = torch.load(working_dir + "de_tokens.pt", weights_only=False)
# en_tokens = torch.load(working_dir + "en_tokens.pt", weights_only=False)

# de_data = torch.load(working_dir + "de_vocab.pt", weights_only=False)
# en_data = torch.load(working_dir + "en_vocab.pt", weights_only=False)

# de_vocab.itos = de_data["itos"]
# de_vocab.stoi = de_data["stoi"]

# en_vocab.itos = en_data["itos"]
# en_vocab.stoi = en_data["stoi"]

In [141]:
print("German vocabulary:", len(de_vocab))
print("English vocabulary:", len(en_vocab))

German vocabulary: 7853
English vocabulary: 5893


In [142]:
# Превращаем предложение в тензор
def numericalize(tokens,vocab):
    ids = [vocab.stoi[BOS]]
    ids += vocab.numericalize(tokens)
    ids += [vocab.stoi[EOS]]
    return [vocab.stoi.get(token,vocab.stoi[UNK]) for token in tokens]

In [171]:
# Создаем PyTorch Dataset
from torch.utils.data import Dataset

class TranslationDataset(Dataset):
    def __init__(self, dataset, de_vocab, en_vocab):
        self.dataset = dataset
        self.de_vocab = de_vocab
        self.en_vocab = en_vocab

    def __len__(self):
        return len(self.dataset)
        
    def __getitem__(self, index):
        example = self.dataset[index]
        de_tokens = tokenize_de(example["de"])
        en_tokens = tokenize_en(example["en"])
        de_ids = numericalize(de_tokens, self.de_vocab)
        en_ids = numericalize(en_tokens, self.en_vocab)
        de_ids = [self.de_vocab.stoi[BOS]] + de_ids + [self.de_vocab.stoi[EOS]]
        en_ids = [self.en_vocab.stoi[BOS]] + en_ids + [self.en_vocab.stoi[EOS]]
        de_tensor = torch.tensor(de_ids,dtype=torch.long)
        en_tensor = torch.tensor(en_ids,dtype=torch.long)
        return de_tensor, en_tensor
    
train_dataset = TranslationDataset(train_data, de_vocab, en_vocab)
valid_dataset = TranslationDataset(valid_data, de_vocab, en_vocab)
test_dataset = TranslationDataset(test_data, de_vocab, en_vocab)

In [173]:
# Решаем проблему разной длины предложений
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    src_batch = []
    trg_batch = []
    
    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)

    src_batch = pad_sequence(src_batch, padding_value=de_vocab.stoi[PAD])
    trg_batch = pad_sequence(trg_batch,padding_value=en_vocab.stoi[PAD])
    return src_batch, trg_batch

In [174]:
# DataLoader
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [177]:
# Позиционное кодирование
import math
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)
        self.register_buffer("pe",pe)
    
    def forward(self, x):
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

In [178]:
class TransformerTranslator(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        trg_vocab_size,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=512,
        dropout=0.1
    ):
        super().__init__()
        self.d_model = d_model

        # Эмбеддинги
        self.src_embedding = nn.Embedding(src_vocab_size,  d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)

        # Позиционное кодирование
        self.src_positional = PositionalEncoding(d_model, dropout)
        self.trg_positional = PositionalEncoding(d_model, dropout)

        # Трансформер
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )

        # Классификатор
        self.fc_out = nn.Linear(d_model, trg_vocab_size)

    def forward(
        self,
        src,
        trg,
        src_mask=None,
        trg_mask=None,
        src_padding_mask=None,
        trg_padding_mask=None,
        memory_key_padding_mask=None
    ):

        # -----------------------------
        # Исходный язык
        # -----------------------------
        src_emb = self.src_embedding(src)
        src_emb = (src_emb * math.sqrt(self.d_model))
        src_emb = self.src_positional(src_emb)

        # -----------------------------
        # Целевой язык
        # -----------------------------
        trg_emb = self.trg_embedding(trg)
        trg_emb = (trg_emb * math.sqrt(self.d_model))
        trg_emb = self.trg_positional(trg_emb)

        # -----------------------------
        # Трансформер
        # -----------------------------
        output = self.transformer(
            src_emb,
            trg_emb,

            src_mask=src_mask,
            tgt_mask=trg_mask,

            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=trg_padding_mask,

            memory_key_padding_mask= memory_key_padding_mask
        )

        # -----------------------------
        # Классификатор
        # -----------------------------
        output = self.fc_out(output)
        return output

In [179]:
# Causal mask
def generate_square_subsequent_mask(size):
    mask = torch.triu(torch.ones(size, size),diagonal=1)
    mask = mask.masked_fill(mask == 1,float("-inf"))
    return mask

# Padding mask
def create_padding_mask(tensor,pad_idx):
    return tensor.transpose(0, 1) == pad_idx

In [180]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerTranslator(
    src_vocab_size=len(de_vocab),
    trg_vocab_size=len(en_vocab),
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=512,
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=en_vocab.stoi[PAD])
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

In [181]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    epoch_loss = 0
    for src, trg in loader:
        src = src.to(device)
        trg = trg.to(device)

        # Decoder input
        trg_input = trg[:-1, :]

        # Target
        trg_output = trg[1:, :]

        # Masks
        trg_mask = generate_square_subsequent_mask(trg_input.size(0)).to(device)
        src_padding_mask = (src.transpose(0, 1)== de_vocab.stoi[PAD])
        trg_padding_mask = (trg_input.transpose(0, 1)== en_vocab.stoi[PAD])

        # Forward
        output = model(
            src, 
            trg_input, 
            trg_mask=trg_mask, 
            src_padding_mask= src_padding_mask,
            trg_padding_mask= trg_padding_mask,
            memory_key_padding_mask= src_padding_mask
        )

        output_dim = output.shape[-1]

        # Flatten
        output = output.reshape(-1, output_dim)

        trg_output = trg_output.reshape(-1)

        # Loss
        loss = criterion(output, trg_output)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [182]:
def evaluate(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, trg in loader:
            src = src.to(device)
            trg = trg.to(device)

            trg_input = trg[:-1, :]
            trg_output = trg[1:, :]

            trg_mask = generate_square_subsequent_mask(trg_input.size(0)).to(device)
            src_padding_mask = (src.transpose(0, 1) == de_vocab.stoi[PAD])
            trg_padding_mask = (trg_input.transpose(0, 1)== en_vocab.stoi[PAD])

            output = model(
                src,
                trg_input,
                trg_mask=trg_mask,
                src_padding_mask= src_padding_mask,
                trg_padding_mask= trg_padding_mask,
                memory_key_padding_mask= src_padding_mask
            )

            output_dim = output.shape[-1]
            output = output.reshape(-1, output_dim)

            trg_output = trg_output.reshape(-1)
            loss = criterion(output, trg_output)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [183]:
# Цикл обучения. ОЧЕНЬ ДОЛГО, особенно если на CPU!
NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    valid_loss = evaluate(
        model,
        valid_loader,
        criterion,
        device
    )

    print(f"Epoch: {epoch + 1}")
    print(f"Train Loss: {train_loss:.3f}")
    print(f"Valid Loss: {valid_loss:.3f}")

C:\Users\D\anaconda3\Lib\site-packages\torch\nn\functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch: 1
Train Loss: 4.041
Valid Loss: 3.202
Epoch: 2
Train Loss: 3.110
Valid Loss: 2.742
Epoch: 3
Train Loss: 2.712
Valid Loss: 2.518
Epoch: 4
Train Loss: 2.438
Valid Loss: 2.306
Epoch: 5
Train Loss: 2.225
Valid Loss: 2.174
Epoch: 6
Train Loss: 2.050
Valid Loss: 2.119
Epoch: 7
Train Loss: 1.901
Valid Loss: 2.064
Epoch: 8
Train Loss: 1.779
Valid Loss: 2.031
Epoch: 9
Train Loss: 1.662
Valid Loss: 1.991
Epoch: 10
Train Loss: 1.569
Valid Loss: 1.977


In [185]:
working_dir = "C:\\Courses\\Innopolis\\2026\\ИИ Инженер\\Models\\"
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),

    "epoch": epoch,

    "src_vocab": de_vocab.stoi,
    "src_itos": de_vocab.itos,

    "trg_vocab": en_vocab.stoi,
    "trg_itos": en_vocab.itos,

    "model_config": {
        "d_model": 256,
        "nhead": 8,
        "num_encoder_layers": 3,
        "num_decoder_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.1,
    }
}, working_dir + "translation_checkpoint.pt")

In [ ]:
# model = TransformerModel(
#     input_dim=len(de_vocab),
#     output_dim=len(en_vocab),
#     d_model=256,
#     nhead=8,
#     num_encoder_layers=3,
#     num_decoder_layers=3,
#     dim_feedforward=512,
#     dropout=0.1
# )

# model.load_state_dict(torch.load(working_dir + "transformer_translation.pt",map_location=device))

# Переводим фразу

In [192]:
def translate_sentence(
    model,
    sentence,
    de_vocab,
    en_vocab,
    device,
    max_length=50
):

    model.eval()
    
    # Исходная фраза
    tokens = tokenize_de(sentence)

    src = numericalize(tokens, de_vocab)
    src = torch.tensor(src)
    src = src.unsqueeze(1).to(device)

    src_padding_mask = (src.transpose(0, 1) == de_vocab.stoi[PAD])

    # Энкодер
    with torch.no_grad():
        src_emb = (model.src_embedding(src) * math.sqrt(model.d_model))
        src_emb = model.src_positional(src_emb )

        memory = model.transformer.encoder(src_emb, src_key_padding_mask= src_padding_mask)

    # Декодер

    trg = torch.tensor([en_vocab.stoi[BOS]], dtype=torch.long, device=device).unsqueeze(1)

    for _ in range(max_length):
        trg_mask = generate_square_subsequent_mask(trg.size(0)).to(device)

        with torch.no_grad():
            trg_emb = (model.trg_embedding(trg) * math.sqrt(model.d_model))
            trg_emb = model.trg_positional(trg_emb)
            output = model.transformer.decoder(trg_emb,memory,tgt_mask=trg_mask,memory_key_padding_mask=src_padding_mask)
            output = model.fc_out(output)

        # Последний токен
        next_token = output[-1].argmax(dim=-1)

        trg = torch.cat([trg,next_token.unsqueeze(0)],dim=0)

        # EOS
        if (next_token.item()== en_vocab.stoi[EOS]):
            break

    # ID в слова

    tokens = [en_vocab.itos[i] for i in trg.squeeze(1).tolist()]
    return tokens

In [193]:
sentence = (
    "Ein Mann sitzt auf einer Bank."
)

tokens = tokenize_de(sentence)
src = numericalize(tokens, de_vocab)

translation = translate_sentence(
    model,
    sentence,
    de_vocab,
    en_vocab,
    device
)

print(translation)

translation = translation[1:-1]

print(" ".join(translation))

['<bos>', 'a', 'man', 'sitting', 'on', 'a', 'bench', 'on', 'a', 'bench', 'on', 'a', 'bench', '.', '<eos>']
a man sitting on a bench on a bench on a bench .


# 4. Предобученный GPT-2

In [ ]:
# Требования
# transformers: 4.57.6
# tokenizers: 0.23.0
# huggingface-hub: 0.36.2

In [219]:
#!pip install "transformers==4.57.6"
#!pip install "tokenizers==0.23.0"
#!pip install "huggingface-hub==0.36.2"

In [32]:
import transformers, huggingface_hub, tokenizers, sys
print(transformers.__version__)
print(huggingface_hub.__version__)
print(tokenizers.__version__)

4.57.6
0.36.2
0.22.2
C:\Users\D\anaconda3\python.exe


In [49]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Режим inference
model.eval()

text = "What is a machine"

# Превращаем текст в token IDs
inputs = tokenizer(text, return_tensors="pt")

print(inputs["input_ids"])

tensor([[2061,  318,  257, 4572]])


In [50]:
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.5,
        top_k=50
    )

result = tokenizer.decode(output[0], skip_special_tokens=True)

print(result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is a machine gun?

A machine gun is a weapon that fires a laser or a laser pointer. It is a small, compact firearm that fires a laser


In [39]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

print("Размер logits:")
print(logits.shape)

Размер logits:
torch.Size([1, 4, 50257])


In [40]:
next_token_logits = logits[:, -1, :]
probs = torch.softmax(next_token_logits,dim=-1)
next_token_id = torch.argmax(probs,dim=-1)
print(next_token_id)
print(tokenizer.decode(next_token_id))

tensor([30])
?


In [44]:
print(probs.sort()[-3:])

(tensor([[8.2949e-13, 1.6411e-12, 2.0229e-12,  ..., 5.5280e-02, 1.1228e-01,
         1.9652e-01]]), tensor([[18361, 48304, 25193,  ...,  4673,   326,    30]]))


In [46]:
tokenizer.decode(4673)

' learning'

# 5. Предобученный BERT 

In [51]:
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [52]:
text = "I really like this movie."
inputs = tokenizer(text, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[ 101, 1045, 2428, 2066, 2023, 3185, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [53]:
with torch.no_grad():
    outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([1, 8, 768])


In [54]:
embeddings = outputs.last_hidden_state
print(embeddings[0, 1])

tensor([ 2.3278e-01, -7.6339e-02, -3.4547e-01, -3.6089e-02,  1.0600e-02,
         1.9353e-01, -4.7421e-01,  1.1021e+00,  7.9790e-02, -5.5660e-01,
        -3.5329e-01, -2.3503e-01,  3.4405e-02,  8.7869e-01,  3.8093e-01,
        -3.9301e-01,  3.9883e-01,  2.3677e-01,  2.4457e-01,  1.1356e+00,
        -2.1338e-01, -1.4986e-01, -8.6431e-01,  1.1268e+00,  3.2524e-01,
        -1.6034e-01, -3.1570e-01, -8.7471e-02, -2.2627e-01, -3.7532e-01,
         2.2778e-02, -4.2104e-01, -1.7946e-01,  4.2449e-01, -3.7124e-01,
        -5.0692e-01, -5.7549e-01, -5.0772e-01, -7.6336e-02,  1.8683e-01,
         5.3932e-02, -4.8061e-01,  1.0852e-01, -5.2600e-01,  7.9636e-01,
        -4.4243e-01,  1.7186e-01, -5.1708e-01,  2.5732e-01, -9.7937e-01,
         2.0584e-01,  2.0285e-01,  6.2613e-01,  4.4154e-01, -1.3036e-01,
         1.1722e+00, -9.9824e-02, -7.4851e-02,  2.9212e-02,  8.7399e-01,
        -9.1157e-02,  4.0675e-01,  1.2632e+00, -6.1039e-01, -1.9591e-01,
         3.2791e-01, -3.9061e-01, -6.9279e-01, -3.3

# 6. BERT для классификации

In [56]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentences = [
    "I really like this movie!",
    "I hate eating oats for breakfast!",
    "Yesterday I had a pleasant four hours long walk"
]

for sentence in sentences:
    result = classifier(sentence)
    print(sentence, ": ", result)

Device set to use cpu


I really like this movie! :  [{'label': 'POSITIVE', 'score': 0.999853253364563}]
I hate eating oats for breakfast! :  [{'label': 'NEGATIVE', 'score': 0.9985595345497131}]
Yesterday I had a pleasant four hours long walk :  [{'label': 'POSITIVE', 'score': 0.9975173473358154}]
